# Data Preparation

## Step 1: Library Import

In [1]:
import pandas as pd
import numpy as np
import os

# path to data folder
data_path = '../data'

# file names
files = {
    'status': 'Participant_Status_13Apr2026.csv',
    'cth': 'FS7_APARC_CTH_13Apr2026.csv',
    'sa': 'FS7_APARC_SA_13Apr2026.csv',
    'aseg': 'FS7_ASEG_VOL_13Apr2026.csv',
    'mriqc': 'MRIQC_13Apr2026.csv'
}

## Step 2: Loading Data

In [2]:
df_status = pd.read_csv(os.path.join(data_path, files['status'])) # master metadata sheet

df_cth = pd.read_csv(os.path.join(data_path, files['cth']))       # cortical thickness
df_sa = pd.read_csv(os.path.join(data_path, files['sa']))         # cortical area
df_aseg = pd.read_csv(os.path.join(data_path, files['aseg']))     # subcortical volumes

df_mriqc = pd.read_csv(os.path.join(data_path, files['mriqc']))   # Magnetic Resonance Imaging Quality Control

print(f"No of subjects in status: {len(df_status)}")

No of subjects in status: 8428


## Step 3: Merging Data

In [3]:
# filter only baseline (BL) visits
df_cth_bl = df_cth[df_cth['EVENT_ID'] == 'BL']
df_sa_bl = df_sa[df_sa['EVENT_ID'] == 'BL']
df_aseg_bl = df_aseg[df_aseg['EVENT_ID'] == 'BL']

# step-by-step merging MRI features
mri_combined = pd.merge(df_cth_bl, df_sa_bl, on=['PATNO', 'EVENT_ID'], suffixes=('_cth', '_sa'))
mri_combined = pd.merge(mri_combined, df_aseg_bl, on=['PATNO', 'EVENT_ID'])

# relevant columns from status
status_subset = df_status[['PATNO', 'COHORT', 'COHORT_DEFINITION', 'ENROLL_AGE', 'ENROLL_STATUS']]

# merge with MRI data
final_df = pd.merge(status_subset, mri_combined, on='PATNO').copy()

# filter out enrolled and complete patients
final_df = final_df[final_df['ENROLL_STATUS'].isin(['Enrolled', 'Complete'])]

# filter PD (1), HC (2), Prodromal (4) - no SWEDD (3)
final_df = final_df[final_df['COHORT'].isin([1, 2, 4])]

# HC=0, PD=1, Prodromal=2
final_df['target'] = final_df['COHORT'].map({2: 0, 1: 1, 4: 2})

print(f"No of subjects after merge: {len(final_df)}")
print(f"\nDistribution:")
print(final_df.groupby(['target', 'COHORT_DEFINITION']).size())

No of subjects after merge: 1236

Distribution:
target  COHORT_DEFINITION  
0       Healthy Control        127
1       Parkinson's Disease    538
2       Prodromal              571
dtype: int64


## Step 4: Cleaning Data

In [4]:
# removing EVENT_ID, COHORT, ENROLL_STATUS -> target i COHORT_DEFINITION
final_df = final_df.drop(columns=['EVENT_ID', 'COHORT', 'ENROLL_STATUS'])

# checking for missing values (NaN)
nan_count = final_df.isnull().sum().sum()
if nan_count > 0:
    print(f"Found {nan_count} empty. Removing rows...")
    final_df = final_df.dropna()

print(f"\nFinal dataset: {len(final_df)} subjects")
print(final_df['COHORT_DEFINITION'].value_counts())

# save to file
final_df.to_csv(os.path.join(data_path, 'processed_data.csv'), index=False)
print("\nDataset saved as 'data/processed_data.csv'")


Final dataset: 1236 subjects
COHORT_DEFINITION
Prodromal              571
Parkinson's Disease    538
Healthy Control        127
Name: count, dtype: int64

Dataset saved as 'data/processed_data.csv'
